# Delta Lake Basics — 09: Partitioning

## What you will learn
Partitioning in Delta works the same as Parquet but with extra features:
partition evolution, partition-level DML, and Liquid Clustering as a modern alternative.

In this notebook:
1. `partitionBy` — choosing and applying partition columns
2. Partition pruning in Delta — verifying it works
3. Partition-level operations — DML, OPTIMIZE, VACUUM
4. Adding/removing partitions dynamically
5. Liquid Clustering — the modern alternative to partitioning
6. Partitioning vs Liquid Clustering — when to use which


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 20:16:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | Delta Lake ready


26/04/10 20:16:51 WARN TaskSetManager: Stage 0 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 20:16:53 WARN TaskSetManager: Stage 1 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.


Dataset ready: 100,000 rows


## Step 1 — partitionBy in Delta


In [2]:
# Partition by category (6 distinct values — ideal cardinality)
TABLE_PART = f"{DATA_DIR}/09_partitioned"
shutil.rmtree(TABLE_PART, ignore_errors=True)

df.write.format("delta") \
        .mode("overwrite") \
        .partitionBy("category") \
        .save(TABLE_PART)

# Show directory structure
import os
print("Directory structure (category partitions):")
for item in sorted(os.listdir(TABLE_PART)):
    if not item.startswith("_") and not item.endswith(".parquet"):
        subdir = f"{TABLE_PART}/{item}"
        files  = [f for f in os.listdir(subdir) if f.endswith(".parquet")]
        rows   = spark.read.format("delta").load(subdir).count()
        size_kb= sum(pathlib.Path(f"{subdir}/{f}").stat().st_size
                     for f in files) // 1024
        print(f"  {item}/  [{len(files)} file(s), {rows:,} rows, {size_kb} KB]")

26/04/10 20:16:55 WARN TaskSetManager: Stage 4 contains a task of very large size (1304 KiB). The maximum recommended task size is 1000 KiB.
                                                                                

Directory structure (category partitions):

If you are trying to read a specific partition, use a where predicate.

CORRECT: spark.read.format("delta").load("/data").where("part=1")
INCORRECT: spark.read.format("delta").load("/data/part=1")
        


26/04/10 20:17:00 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  category=Books/  [4 file(s), 16,905 rows, 372 KB]

If you are trying to read a specific partition, use a where predicate.

CORRECT: spark.read.format("delta").load("/data").where("part=1")
INCORRECT: spark.read.format("delta").load("/data/part=1")
        
  category=Clothing/  [4 file(s), 16,743 rows, 373 KB]

If you are trying to read a specific partition, use a where predicate.

CORRECT: spark.read.format("delta").load("/data").where("part=1")
INCORRECT: spark.read.format("delta").load("/data/part=1")
        
  category=Electronics/  [4 file(s), 16,496 rows, 372 KB]

If you are trying to read a specific partition, use a where predicate.

CORRECT: spark.read.format("delta").load("/data").where("part=1")
INCORRECT: spark.read.format("delta").load("/data/part=1")
        
  category=Food/  [4 file(s), 16,487 rows, 364 KB]

If you are trying to read a specific partition, use a where predicate.

CORRECT: spark.read.format("delta").load("/data").where("part=1")
INCORRECT: spark.read.fo

## Step 2 — Partition Pruning Verification


In [3]:
runs = 3

# Without partition filter — full scan
def full_scan():
    return spark.read.format("delta").load(TABLE_PART).agg(F.sum("revenue")).collect()

# With partition filter — pruning
def pruned_scan():
    return spark.read.format("delta").load(TABLE_PART) \
               .filter(F.col("category") == "Electronics") \
               .agg(F.sum("revenue")).collect()

tf, tp = [], []
for _ in range(runs):
    t0=time.time(); full_scan(); tf.append(time.time()-t0)
    t0=time.time(); pruned_scan(); tp.append(time.time()-t0)

print(f"Full scan (all categories): {sorted(tf)[1]:.3f}s")
print(f"Partition pruning (1 of 6) : {sorted(tp)[1]:.3f}s")
print(f"Speedup                     : {sorted(tf)[1]/sorted(tp)[1]:.1f}x")
print()
print("Explain — verify PartitionFilters:")
spark.read.format("delta").load(TABLE_PART) \
     .filter(F.col("category") == "Electronics") \
     .explain(mode="simple")

Full scan (all categories): 0.806s
Partition pruning (1 of 6) : 0.480s
Speedup                     : 1.7x

Explain — verify PartitionFilters:
== Physical Plan ==
*(1) Project [order_id#2621L, customer_id#2622L, product#2623, category#2624, country#2625, quantity#2626, price#2627, revenue#2628, status#2629, order_date#2630]
+- *(1) ColumnarToRow
   +- FileScan parquet [order_id#2621L,customer_id#2622L,product#2623,country#2625,quantity#2626,price#2627,revenue#2628,status#2629,order_date#2630,category#2624] Batched: true, DataFilters: [], Format: Parquet, Location: PreparedDeltaFileIndex(1 paths)[file:/workspace/data/delta_basics/09_partitioned], PartitionFilters: [isnotnull(category#2624), (category#2624 = Electronics)], PushedFilters: [], ReadSchema: struct<order_id:bigint,customer_id:bigint,product:string,country:string,quantity:int,price:double...




## Step 3 — Partition-Level Operations


In [4]:
# DML on specific partition only
before = spark.read.format("delta").load(TABLE_PART) \
              .filter(F.col("category")=="Electronics") \
              .filter(F.col("status")=="pending").count()

spark.sql(f"""
    UPDATE delta.`{TABLE_PART}`
    SET status = 'confirmed'
    WHERE category = 'Electronics' AND status = 'pending'
""")
# Delta is smart: only rewrites Electronics partition files!

after = spark.read.format("delta").load(TABLE_PART) \
             .filter(F.col("category")=="Electronics") \
             .filter(F.col("status")=="confirmed").count()
print(f"UPDATE in Electronics partition:")
print(f"  pending → confirmed: {before:,} rows updated (only Electronics files rewritten)")

# OPTIMIZE only a specific partition
spark.sql(f"""
    OPTIMIZE delta.`{TABLE_PART}`
    WHERE category = 'Books'
""")
print("\nOPTIMIZE only Books partition ✅ (other partitions untouched)")

# Delete entire partition efficiently
spark.sql(f"""
    DELETE FROM delta.`{TABLE_PART}`
    WHERE category = 'Furniture'
""")
remaining_cats = spark.read.format("delta").load(TABLE_PART) \
                      .select("category").distinct().count()
print(f"\nAfter DELETE Furniture: {remaining_cats} categories remaining")

26/04/10 20:17:17 WARN UpdateCommand: Could not validate number of records due to missing statistics.
                                                                                

UPDATE in Electronics partition:
  pending → confirmed: 3,263 rows updated (only Electronics files rewritten)

OPTIMIZE only Books partition ✅ (other partitions untouched)

After DELETE Furniture: 5 categories remaining


## Step 4 — Liquid Clustering: Modern Alternative


In [5]:
# Liquid Clustering: no static partitions
# OPTIMIZE incrementally re-clusters changed files
TABLE_LC = f"{DATA_DIR}/09_liquid"
shutil.rmtree(TABLE_LC, ignore_errors=True)

spark.sql(f"""
    CREATE TABLE delta.`{TABLE_LC}`
    USING delta
    CLUSTER BY (category, country)
    AS SELECT * FROM delta.`{TABLE_PART}`
""")

print(f"Liquid Clustering table created: {spark.read.format('delta').load(TABLE_LC).count():,} rows")
print()

# Check clustering columns
detail = DeltaTable.forPath(spark, TABLE_LC).detail().collect()[0]
# detail() returns a Row — use hasattr or try/except, not .get()
clustering = getattr(detail, 'clusteringColumns', None)
print(f"Clustering columns: {clustering if clustering is not None else 'not shown in this version'}")

# Append new data — clustering is NOT applied automatically
extra = df.limit(5000)
extra.write.format("delta").mode("append").save(TABLE_LC)
print(f"After append: {spark.read.format('delta').load(TABLE_LC).count():,} rows")

# Run OPTIMIZE to apply clustering to new files
print("\nRunning OPTIMIZE to apply liquid clustering...")
spark.sql(f"OPTIMIZE delta.`{TABLE_LC}`")
print("Clustering applied incrementally to new files ✅")

Liquid Clustering table created: 83,422 rows

Clustering columns: ['category', 'country']


26/04/10 20:17:28 WARN TaskSetManager: Stage 164 contains a task of very large size (1252 KiB). The maximum recommended task size is 1000 KiB.


After append: 88,422 rows

Running OPTIMIZE to apply liquid clustering...
Clustering applied incrementally to new files ✅


## Summary: Partitioning vs Liquid Clustering

| | Traditional Partitioning | Liquid Clustering |
|---|---|---|
| Setup | `partitionBy("col")` | `CLUSTER BY (col1, col2)` |
| Directory structure | Yes — `category=Electronics/` | No — single flat directory |
| Cardinality limit | < 1000 values (else small files) | No limit |
| Re-clustering cost | Must rewrite ALL data | Incremental via OPTIMIZE |
| Multi-column | Yes (nested dirs) | Yes (Z-order aware) |
| Best for | Stable, low-cardinality columns | Any column, changing patterns |

### When to use each
```
Use partitionBy when:
  - Column has < 100-1000 distinct values
  - You always filter on this column
  - Values are stable (not growing cardinality)
  - e.g.: year/month/date, region, category

Use Liquid Clustering when:
  - High-cardinality columns (user_id, product_id)
  - Multiple filter columns with different patterns
  - You want to change clustering columns later easily
  - New tables — Liquid Clustering is the recommended default
```
